[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_00/MFM_ch0_passive_elasticity/MFM_ch0_passive_elasticity.ipynb)

# MFM_ch0_passive_elasticity

**Modelling Financial Markets (MFM) — Chapter 0: Financial Markets in 2026**

Case study Haddad, Huebner and Loualiche (2025, American Economic Review), How Competitive Is the Stock Market?: the share of exchange-traded funds in US corporate equities (Financial Accounts of the United States, quarterly, 2001 Q1 to the latest quarter) and the implied fall in the active share and in the aggregate demand elasticity under the rule of Section V.A of the paper (81% active investors in 2001, pass-through 0.33); and the aggregate elasticity of eq. (5) as a function of the share of investors turned passive, for a degree of strategic response of 0, 2.97 (Table 2) and 10.

*Author: Daniel Traian Pele*

In [ ]:
import os
import io
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

## Style and data loader

In [ ]:
# Stil standard MFM (identic cu SFM): transparent + ENG + legenda jos
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Culori brand
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Gray     = '#7F7F7F'
LightGray = '#DADADA'
Teal     = '#17A2B8'


def save_fig(name):
    """Salveaza figura ca PDF si PNG transparent, in folderul curent."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Plaseaza legenda in afara graficului, jos-centru."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def bottom_legend(fig, ncol=3, handles=None, labels=None, fontsize=8):
    """Legenda sub figura (in afara axelor), dupa tight_layout; save_fig o include (bbox_inches='tight')."""
    plt.tight_layout()
    if handles is None:
        handles, labels, seen = [], [], set()
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in seen and not l.startswith('_'):
                    handles.append(h); labels.append(l); seen.add(l)
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.0), ncol=ncol, frameon=False, fontsize=fontsize)


def log_axis(ax, ticks):
    """Axa logaritmica cu etichete lizibile (fara notatie stiintifica)."""
    ax.set_yscale('log')
    ax.set_yticks(ticks)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:g}'))
    ax.yaxis.set_minor_formatter(plt.NullFormatter())


In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
LOCAL_DIRS = ['../../../data/market', '../../data/market', '../data/market', 'data/market']   # repo clonat
START, END = '2000-01-01', '2026-09-18'

# nume -> (sursa, simbol, camp)
SERIES = {
    'S&P 500': ('market', 'GSPC.INDX', 'close'),
    'Euro Stoxx 50': ('market', 'STOXX50E.INDX', 'close'),
    'Nikkei 225': ('market', 'N225.INDX', 'close'),
    'VIX': ('market', 'VIX.INDX', 'close'),
    'BET-TR': ('market', 'BETTR.INDX', 'close'),
    'BET': ('market', 'BET', 'close'),
    'BET-XT': ('market', 'BETXT.INDX', 'close'),
    'US Treasuries 20y+ (TLT)': ('market', 'TLT.US', 'adjusted_close'),
    'SPY': ('market', 'SPY.US', 'adjusted_close'),
    'RSP': ('market', 'RSP.US', 'adjusted_close'),
    'IBIT': ('market', 'IBIT.US', 'close'),
    'IBIT volume': ('market', 'IBIT.US', 'volume'),
    'Gold': ('market', 'XAUUSD.FOREX', 'close'),
    'EUR/USD': ('market', 'EURUSD.FOREX', 'close'),
    'EUR/RON (EODHD)': ('market', 'EURRON.FOREX', 'close'),
    'Bitcoin': ('market', 'BTC-USD.CC', 'close'),
    'Ethereum': ('market', 'ETH-USD.CC', 'close'),
    'Tether (USDT)': ('market', 'USDT-USD.CC', 'close'),
    'Romania 10y': ('market', 'RO10Y.GBOND', 'close'),
    'Germany 10y': ('market', 'DE10Y.GBOND', 'close'),
    'Banca Transilvania': ('market', 'TLV.RO', 'adjusted_close'),
    'OMV Petrom': ('market', 'SNP.RO', 'adjusted_close'),
    'BRD': ('market', 'BRD.RO', 'adjusted_close'),
    'Transgaz': ('market', 'TGN.RO', 'adjusted_close'),
    'Romgaz': ('market', 'SNG.RO', 'adjusted_close'),
    'Hidroelectrica': ('market', 'H2O.RO', 'adjusted_close'),
    'EUR/RON': ('bnr', 'EUR', None),
    'DGS10': ('fred', 'DGS10', None),
    'DGS2': ('fred', 'DGS2', None),
    'FEDFUNDS': ('fred', 'FEDFUNDS', None),
    'USD per EUR': ('fred', 'DEXUSEU', None),
    'JPY per USD': ('fred', 'DEXJPUS', None),
    'T10Y2Y': ('fred', 'T10Y2Y', None),
    'ETF equities': ('fred', 'BOGZ1LM563064100Q', None),
    'All equities': ('fred', 'BOGZ1LM893064105Q', None),
    'Stablecoins': ('defillama', 'stablecoincharts/all', None),
}
WEEKDAYS_ONLY = {'Gold', 'EUR/USD'}


def read_market(symbol):
    """Fisierul de date de piata: local (repo clonat), altfel din repo-ul GitHub."""
    for d in LOCAL_DIRS:
        path = os.path.join(d, f'{symbol}.csv')
        if os.path.exists(path):
            return pd.read_csv(path, index_col='date', parse_dates=True)
    return pd.read_csv(REPO_RAW + f'{symbol}.csv', index_col='date', parse_dates=True)


def fetch_daily(symbol, source='market', field='close', start=START, end=END):
    """Punctul unic de acces la date: intoarce o pd.Series zilnica (fara NaN)."""
    if source == 'market':
        s = read_market(symbol)[field]
    elif source == 'fred':
        raw = urllib.request.urlopen(
            f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={symbol}', timeout=90).read().decode()
        s = pd.read_csv(io.StringIO(raw), index_col=0, parse_dates=True, na_values='.').iloc[:, 0]
    elif source == 'defillama':
        req = urllib.request.Request(f'https://stablecoins.llama.fi/{symbol}',
                                     headers={'User-Agent': 'Mozilla/5.0'})
        d = json.load(urllib.request.urlopen(req, timeout=90))
        s = pd.Series({pd.to_datetime(int(x['date']), unit='s'): x['totalCirculatingUSD'].get('peggedUSD', 0)
                       for x in d}) / 1e9                                   # miliarde USD
    elif source == 'bnr':
        # cursul oficial BNR (RON pentru o unitate de valuta), arhive XML anuale
        import re
        rows = []
        for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
            url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
            xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                         timeout=90).read().decode()
            for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
                m = re.search(rf'<Rate currency="{symbol}">([\d.]+)</Rate>', body)
                if m:
                    rows.append((d, float(m.group(1))))
        s = pd.Series(dict(rows))
    else:
        raise ValueError(f'unknown source: {source}')
    s = pd.to_numeric(s, errors='coerce')
    s.index = pd.to_datetime(s.index)
    s.index.name = 'Date'
    return s.sort_index().loc[start:end].dropna()


def load(name, start=START, end=END):
    """Seria cu numele din SERIES (pentru aur si EUR/USD: doar zilele lucratoare)."""
    source, symbol, field = SERIES[name]
    s = fetch_daily(symbol, source, field, start, end).rename(name)
    if name in WEEKDAYS_ONLY:
        s = s[s.index.dayofweek < 5]
    return s


def load_panel(names, start=START, end=END):
    """Mai multe serii aliniate pe data (NaN acolo unde o piata nu tranzactioneaza)."""
    return pd.concat([load(n, start, end) for n in names], axis=1)


## Helper functions

In [ ]:
HHL = {'active0': 0.81, 'chi': 2.97, 'pass_through': 0.33}

def etf_share():
    """Ponderea ETF-urilor in actiunile corporative SUA (Financial Accounts of the US), trimestrial, 2001 Q1 - ultimul trimestru."""
    q = load_panel(['ETF equities', 'All equities'], start='2001-01-01').dropna()
    return (q['ETF equities'] / q['All equities']).rename('ETF share')

## Data

In [ ]:
s = etf_share()
print(s.tail())

## Charts

In [ ]:
def fig_hhl_passive():
    """Stanga: ponderea ETF. Dreapta: regula din Sectiunea V.A (81% activi la start, transmisie 0,33):
    scaderea ponderii active si scaderea implicita a elasticitatii agregate E_agg fata de 2001 Q1."""
    s = etf_share()
    active = HHL['active0'] - (s - s.iloc[0])
    d_active = active / HHL['active0'] - 1
    d_elast = HHL['pass_through'] * d_active
    fig, axes = plt.subplots(1, 2, figsize=(6.6, 2.35))
    axes[0].plot(s.index, 100 * s.values, color=Teal, lw=1.1, label='ETF share of US corporate equities')
    for d in [s.index[0], pd.Timestamp('2020-10-01'), s.index[-1]]:
        axes[0].plot(d, 100 * s.loc[d], 'o', ms=3.5, color=Teal)
        axes[0].annotate(f'{100 * s.loc[d]:.1f}%', xy=(d, 100 * s.loc[d]), xytext=(0, 6),
                         textcoords='offset points', ha='center', fontsize=7, color='black')
    axes[0].set_ylabel('Share of market value (%)')
    axes[0].set_ylim(0, 12.5)
    axes[0].set_xlim(pd.Timestamp('2000-01-01'), pd.Timestamp('2027-12-31'))
    axes[0].set_title('ETF share, 2001 Q1 - ' + f'{s.index[-1].year} Q{s.index[-1].quarter}', fontsize=9, loc='left')
    axes[1].plot(d_active.index, 100 * d_active.values, color=IDAred, lw=1.1,
                 label='Active share = elasticity if $\\chi = 0$')
    axes[1].plot(d_elast.index, 100 * d_elast.values, color=MainBlue, lw=1.1,
                 label='Elasticity, pass-through 0.33 ($\\chi = 2.97$)')
    axes[1].axhline(0, color=Gray, lw=0.5, ls=':')
    for d in [pd.Timestamp('2020-10-01'), s.index[-1]]:
        for ser, c in [(d_active, IDAred), (d_elast, MainBlue)]:
            axes[1].plot(d, 100 * ser.loc[d], 'o', ms=3.5, color=c)
            axes[1].annotate(f'{100 * ser.loc[d]:.1f}%', xy=(d, 100 * ser.loc[d]), xytext=(-4, -9),
                             textcoords='offset points', ha='right', fontsize=7, color='black')
    axes[1].set_ylabel('Change since 2001 Q1 (%)')
    axes[1].set_ylim(-15, 1.5)
    axes[1].set_title('Implied change (81% active in 2001)', fontsize=9, loc='left')
    for ax in axes:
        ax.xaxis.set_major_locator(mdates.YearLocator(5))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
        ax.tick_params(axis='x', labelsize=7.5)
    bottom_legend(fig, ncol=3, fontsize=7.5)
    save_fig('ch0_hhl_passive')
    pick = lambda d: dict(etf=round(s.loc[d], 4), active=round(d_active.loc[d], 4), elast=round(d_elast.loc[d], 4))
    return {str(d.date()): pick(d) for d in [s.index[0], pd.Timestamp('2020-10-01'), s.index[-1]]}

In [ ]:
def fig_hhl_elasticity():
    """Ec. (5): elasticitatea agregata dupa ce o fractie 1 - alpha din investitori devine pasiva,
    relativ la piata fara investitori pasivi: alpha (1 + chi) / (1 + chi alpha)."""
    alpha = np.linspace(0, 1, 401)
    rel = lambda chi: alpha * (1 + chi) / (1 + chi * alpha)
    fig, ax = plt.subplots(figsize=(4.7, 3.0))
    passive = 100 * (1 - alpha)
    for chi, c, lab in [(0, IDAred, '$\\chi = 0$: no strategic response (Koijen and Yogo)'),
                        (HHL['chi'], MainBlue, '$\\chi = 2.97$: estimate (Table 2, row 1)'),
                        (10, Forest, '$\\chi = 10$: strong response')]:
        ax.plot(passive, rel(chi), color=c, lw=1.3, label=lab)
    ax.axvline(30, color=Gray, lw=0.5, ls=':')
    for chi, c, dy in [(0, IDAred, -5), (HHL['chi'], MainBlue, -5), (10, Forest, 5)]:
        v = 0.70 * (1 + chi) / (1 + chi * 0.70)
        ax.plot(30, v, 'o', ms=4, color=c)
        ax.annotate(f'{v:.2f}', xy=(30, v), xytext=(-6, dy), textcoords='offset points', ha='right',
                    va='top' if dy < 0 else 'bottom', fontsize=7.5, color='black')
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 1.1)
    ax.set_xlabel('Share of investors turned passive, $1 - \\alpha$ (%)')
    ax.set_ylabel('$\\mathcal{E}_{agg}$ relative to all active')
    ax.set_title('Aggregate elasticity when investors turn passive, eq. (5)', fontsize=9, loc='left')
    bottom_legend(fig, ncol=1)
    save_fig('ch0_hhl_elasticity')
    return {chi: round(0.70 * (1 + chi) / (1 + chi * 0.70), 3) for chi in (0, HHL['chi'], 10)}

## Run

In [ ]:
print(fig_hhl_passive())
print(fig_hhl_elasticity())

![ch0_hhl_passive](./ch0_hhl_passive.png)

![ch0_hhl_elasticity](./ch0_hhl_elasticity.png)

Output: `ch0_hhl_passive.pdf`, `ch0_hhl_elasticity.pdf`